# Tier 1.2 — category-aux (cat_w) ablation at train_limit=2000
Stream once, train 3 configs back-to-back (decay 0.999/0.99/0.995, rampup 5), sweep each on identical val. Needs GPU ON, Internet ON, Secret **HF_TOKEN**. Run via **Save & Run All (Commit)** (~1h).


In [ ]:
import os, sys
_MODULES = {
  'config.py': (
  "\"\"\"Configuration and global constants for the v-3 WavLM SED baseline.\n\nThis is the single source of truth for every tunable in the pipeline: the budget\nswitch, the audio/training h"
  "yper-parameters, the category vocabulary, the dataset\nlocation and the derived frame-grid constants that the rest of the package imports.\n\nNothing here trains or touches the networ"
  "k - it only defines constants and a couple\nof tiny seeding helpers, so it is safe to import from anywhere without side effects\nbeyond RNG seeding when you call `seed_everything()`."
  "\n\"\"\"\n\nfrom __future__ import annotations\n\nimport random\nfrom pathlib import Path\n\nimport numpy as np\n\n\n# ---------------------------------------------------------------------------"
  "\n# Budget switch - the one knob you actually touch.\n#   \"fast\" : ~3 h smoke/dev run with small per-tier caps and few epochs.\n#   \"full\" : ~7 h full run using far more clips and mor"
  "e epochs.\n# ---------------------------------------------------------------------------\nBUDGET = \"fast\"\n\n\n# ------------------------------------------------------------------------"
  "---\n# Core audio + training config. Consumed all over the package via CFG[...].\n# ---------------------------------------------------------------------------\nCFG = dict(\n    sr=160"
  "00,             # WavLM expects 16 kHz mono audio\n    hop=160,              # 160 / 16000 = 10 ms label grid\n    clip_sec=5.0,         # 5 s window: halves the WavLM sequence, doub"
  "les the batch\n    time_pool=2,          # pool 10 ms -> 20 ms, matching WavLM's native 50 Hz output\n    n_cat=7,              # number of noise categories (channel 0 is \"any noise\""
  ")\n    silver_weight=0.4,    # Silver-tier labels count 40% of Gold\n    ema_decay=0.999,      # mean-teacher EMA decay\n    max_cons_w=2.0,       # max consistency weight after ramp-"
  "up\n    rampup_epochs=5,      # consistency ramp-up length in epochs\n    pos_weight=4.0,       # up-weight rare noise frames in BCE\n    cat_w=1.0,              # weight of auxiliary"
  " category channels 1..7 in\n                            # l_strong/l_weak (ch0 any-noise always 1.0).\n                            # 1.0 = current behaviour; ablate with 0.5 / 0.0.\n "
  "   mixup_prob=0.5,       # probability of applying mixup to a batch\n    mixup_alpha=0.2,      # Beta(alpha, alpha) mixing coefficient\n    val_frac=0.1,         # 10% of gold clips "
  "held out for validation\n    weight_decay=1e-2,\n    num_workers=2,\n    seed=42,\n)\n\n\n# ---------------------------------------------------------------------------\n# Categories. Predi"
  "cted internally (channels 1..7); Track 1 submits only the\n# class-agnostic \"any noise\" channel 0, so these are for training signal only.\n# -----------------------------------------"
  "----------------------------------\nCATS = [\n    \"animal\", \"vehicle_traffic\", \"baby_child\", \"singing_music\",\n    \"phone_signal_alarm\", \"appliance_machine\", \"human_non_speech\",\n]\nCAT"
  "2IDX = {c: i for i, c in enumerate(CATS)}\nN_OUT = 1 + CFG[\"n_cat\"]   # 8 output channels: [any-noise, cat0..cat6]\n\n\n# --------------------------------------------------------------"
  "-------------\n# Dataset + output locations (Kaggle-style paths).\n# ---------------------------------------------------------------------------\nREPO = \"ARTPARK-IISc/Vaani-Noise-Even"
  "t-Dataset\"\nWORK = Path(\"/kaggle/working\")\nCKPT = WORK / \"t1_wavlm.pt\"\n\n\n# ---------------------------------------------------------------------------\n# Encoder settings.\n#   DROP_L"
  "AST_LAYERS: Schmid et al. (ICASSP 2025) found the final WavLM layers\n#   overfit the pretraining task while earlier layers transfer better, so we drop\n#   the last two transformer "
  "blocks.\n# ---------------------------------------------------------------------------\nENCODER_NAME = \"microsoft/wavlm-base-plus\"\nDROP_LAST_LAYERS = 2\nLR_ENC, LR_HEAD = 5e-5, 1e-3\n\n"
  "\n# ---------------------------------------------------------------------------\n# Derived constants used everywhere else (window / frame sizes).\n# ----------------------------------"
  "-----------------------------------------\nN_SAMP = int(CFG[\"clip_sec\"] * CFG[\"sr\"])                 # samples in a 5 s window\nFRAMES_100HZ = int(CFG[\"clip_sec\"] * CFG[\"sr\"] / CFG[\""
  "hop\"])   # 10 ms frames per window\nFRAMES_OUT = FRAMES_100HZ // CFG[\"time_pool\"]             # 20 ms output frames per window\nFRAME_SEC = CFG[\"hop\"] * CFG[\"time_pool\"] / CFG[\"sr\"] "
  "    # seconds per (20 ms) output frame\n\n# WavLM conv front-end needs >= ~400 samples; keep a margin for very short clips.\nMIN_SAMPLES = 640\n# Longest clip we run through the model "
  "in a single forward pass. Above this we\n# window the clip to bound memory (see inference.t1_posteriors).\nMAX_ONESHOT_SEC = 120.0\n\n\ndef budget_settings(budget: str = BUDGET, train_l"
  "imit: int | None = None):\n    \"\"\"Return the (MAX_PER_QUALITY, STAGE1_EPOCHS, STAGE2_EPOCHS, T1_BS, TRAIN_LIMIT)\n    tuple for the requested budget.\n\n    `MAX_PER_QUALITY` caps how "
  "many clips of each annotation tier we decode. When\n    `train_limit` is small, decoding thousands of clips is wasteful, so the caps are\n    tightened to just above the limit (plus "
  "headroom for the gold-only val split).\n    \"\"\"\n    if budget == \"fast\":\n        max_per_quality = {\"verified_timestamps\": 6000, \"unverified_timestamps\": 6000,\n                     "
  "      \"no_timestamps\": 2000}\n        stage1_epochs, stage2_epochs, t1_bs = 3, 8, 16\n        default_limit = 100          # smoke-run cap on training clips\n    else:\n        max_per"
  "_quality = {\"verified_timestamps\": None, \"unverified_timestamps\": 20000,\n                           \"no_timestamps\": 8000}\n        stage1_epochs, stage2_epochs, t1_bs = 6, 16, 16\n "
  "       default_limit = None         # use everything\n\n    limit = default_limit if train_limit is None else train_limit\n    if limit is not None:\n        cap = limit + 32          "
  "   # headroom for the gold-only validation split\n        max_per_quality = {k: (min(v, cap) if v is not None else cap)\n                           for k, v in max_per_quality.items("
  ")}\n    return max_per_quality, stage1_epochs, stage2_epochs, t1_bs, limit\n\n\ndef seed_everything(seed: int = CFG[\"seed\"]):\n    \"\"\"Seed Python and NumPy RNGs. Torch is seeded separat"
  "ely in data.py once torch\n    is imported, so this module stays import-light.\"\"\"\n    random.seed(seed)\n    np.random.seed(seed)\n\n\n# Seed the light RNGs at import time so constant-t"
  "ime helpers are reproducible.\nseed_everything()\n"
),
  'data.py': (
  "\"\"\"Data loading, label rasterisation and the SED dataset for the v-3 WavLM baseline.\n\nThe flow is:\n  1. `load_stream()` opens the HF dataset as a stream with `decode=False`.\n  2. `"
  "build_rows()` makes a single pass over the stream, decoding each kept clip to\n     an int16 waveform in memory and rasterising its labels to a 100 Hz grid.\n  3. `bucket_by_tier()` "
  "splits those rows into gold/silver/bronze.\n  4. `SEDData` crops fixed 5 s windows and returns training tensors.\n  5. `build_dataloaders()` produces the train/val loaders (val is go"
  "ld-only).\n\nEverything stays in memory - there is no disk cache - which matches the reference\nscript and keeps the pipeline simple.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport i"
  "o\nimport math\nimport random\nimport time\nfrom collections import defaultdict\n\nimport numpy as np\nimport torch\nimport librosa\nimport soundfile as sf\nfrom torch.utils.data import Data"
  "set, DataLoader\n\nfrom config import (\n    CFG, CAT2IDX, N_OUT, N_SAMP, FRAMES_100HZ, FRAMES_OUT, FRAME_SEC, REPO,\n)\n\n\n# ------------------------------------------------------------"
  "---------------\n# Torch seeding. random/np are seeded in config; torch seeds model init and the\n# DataLoader sampler, which otherwise start from OS entropy.\n# ---------------------"
  "------------------------------------------------------\ndef seed_torch(seed: int = CFG[\"seed\"]):\n    \"\"\"Seed torch (CPU + CUDA). DataLoader workers inherit from this seed.\"\"\"\n    to"
  "rch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\n# ---------------------------------------------------------------------------\n# Audio decoding\n# ----------------------"
  "-----------------------------------------------------\ndef decode_audio(a, target_sr: int = CFG[\"sr\"]):\n    \"\"\"Decode one `audio` cell to a mono float32 waveform at `target_sr`.\n\n  "
  "  Handles every shape the `datasets` Audio feature has ever returned: a dict with\n    `array`/`bytes`/`path`, a torchcodec `AudioDecoder`, or an object with `.path`.\n    We always "
  "decode ourselves (config passes decode=False) because the library's\n    decoder has changed twice and only raw bytes are version-proof.\n    \"\"\"\n    w = sr = None\n    if isinstance("
  "a, dict):\n        if a.get(\"array\") is not None:\n            w, sr = np.asarray(a[\"array\"], dtype=np.float32), a[\"sampling_rate\"]\n        elif a.get(\"bytes\"):\n            w, sr = s"
  "f.read(io.BytesIO(a[\"bytes\"]), dtype=\"float32\", always_2d=False)\n        elif a.get(\"path\"):\n            w, sr = sf.read(a[\"path\"], dtype=\"float32\", always_2d=False)\n    elif hasat"
  "tr(a, \"get_all_samples\"):\n        s = a.get_all_samples()\n        w, sr = s.data.numpy().astype(np.float32), int(s.sample_rate)\n    elif hasattr(a, \"path\"):\n        w, sr = sf.read"
  "(a.path, dtype=\"float32\", always_2d=False)\n    if w is None:\n        raise ValueError(f\"cannot decode {type(a)}\")\n    w = np.asarray(w, dtype=np.float32)\n    if w.ndim > 1:        "
  "                          # downmix stereo -> mono\n        w = w.mean(axis=0) if w.shape[0] < w.shape[1] else w.mean(axis=1)\n    if sr != target_sr:\n        w = librosa.resample(w,"
  " orig_sr=sr, target_sr=target_sr)\n    return np.ascontiguousarray(w, dtype=np.float32)\n\n\ndef _f(x):\n    \"\"\"Best-effort float parse; returns None on failure (bad timestamp strings)."
  "\"\"\"\n    try:\n        return float(x)\n    except Exception:\n        return None\n\n\ndef spans_from(ex):\n    \"\"\"Extract sorted (start, end, category) spans from a row's timestamp field"
  ".\n\n    Bronze clips legitimately have an empty list here - that is correct, not a\n    parsing failure.\n    \"\"\"\n    out = []\n    for s in (ex.get(\"NoiseSubCategoryTimeStamp\") or [])"
  ":\n        st, en = _f(s.get(\"start\")), _f(s.get(\"end\"))\n        if st is not None and en is not None and en > st:\n            out.append((st, en, s.get(\"category\")))\n    return sor"
  "ted(out)\n\n\ndef build_labels(spans, n_frames, has_strong):\n    \"\"\"Rasterise timestamp spans onto a 100 Hz `(N_OUT, n_frames)` label grid.\n\n    Channel 0 = \"any noise\", channels 1..7"
  " = the per-category masks. When\n    `has_strong` is False (Bronze) the array stays all zeros because there is no\n    timing to rasterise.\n    \"\"\"\n    lab = np.zeros((N_OUT, n_frame"
  "s), dtype=np.uint8)\n    if has_strong:\n        for st, en, cat in spans:\n            a, b = max(0, int(round(st * 100))), min(n_frames, int(round(en * 100)))\n            if b <= a:"
  "\n                continue\n            lab[0, a:b] = 1\n            ci = CAT2IDX.get(cat)\n            if ci is not None:\n                lab[1 + ci, a:b] = 1\n    return lab\n\n\ndef cli"
  "p_tags(ex):\n    \"\"\"Build the clip-level `(N_OUT,)` tag vector from `NoiseCategory`.\n\n    Present on every tier including Bronze; channel 0 flips on when any category is\n    present"
  ". This is the only supervision Bronze clips carry.\n    \"\"\"\n    y = np.zeros(N_OUT, dtype=np.float32)\n    cats = ex.get(\"NoiseCategory\") or []\n    if len(cats):\n        y[0] = 1.0\n "
  "   for c in cats:\n        ci = CAT2IDX.get(c)\n        if ci is not None:\n            y[1 + ci] = 1.0\n    return y\n\n\n# --------------------------------------------------------------"
  "-------------\n# Streaming load + single decode pass\n# ---------------------------------------------------------------------------\ndef load_stream(hf_token: str | None = None):\n    "
  "\"\"\"Open the dataset as a decode=False stream so we pull rows lazily and stop as\n    soon as the per-tier caps are met (never downloading shards we won't use).\"\"\"\n    from datasets "
  "import load_dataset, Audio\n    raw = load_dataset(REPO, split=\"train\", streaming=True, token=hf_token)\n    raw = raw.cast_column(\"audio\", Audio(decode=False))\n    return raw\n\n\ndef "
  "build_rows(raw, max_per_quality, verbose: bool = True):\n    \"\"\"Single pass over the stream: decode each kept clip once and rasterise labels.\n\n    Returns a list of dict rows, each "
  "with:\n      wav      - int16 waveform (keeps RAM small; WavLM consumes raw audio)\n      lab      - (N_OUT, T) 100 Hz label mask\n      tags     - (N_OUT,) clip-level tags\n      qual"
  "ity  - annotation tier string\n      has_strong - True for gold/silver (timestamped), False for bronze\n    \"\"\"\n    from tqdm.auto import tqdm\n\n    counts_done = {k: 0 for k in max_p"
  "er_quality}\n\n    def done():\n        # Finished once every capped tier has hit its cap (None means \"no cap\").\n        for q, cap in max_per_quality.items():\n            if cap is N"
  "one:\n                return False\n            if cap and counts_done[q] < cap:\n                return False\n        return True\n\n    rows, t0 = [], time.time()\n    for i, ex in enu"
  "merate(tqdm(raw, desc=\"loading\")):\n        q = ex[\"annotationQuality\"]\n        cap = max_per_quality.get(q, 0)\n        if cap == 0:\n            continue\n        if cap is not None "
  "and counts_done[q] >= cap:\n            if done():\n                break\n            continue\n        try:\n            wav = decode_audio(ex[\"audio\"])\n        except Exception as e:"
  "\n            if i < 5:\n                print(\"decode failed row\", i, e)\n            continue\n        if wav is None or len(wav) < CFG[\"sr\"] * 0.2:   # skip <200 ms junk\n           "
  " continue\n        spans = spans_from(ex)\n        has_strong = q in (\"verified_timestamps\", \"unverified_timestamps\")\n        nf = 1 + len(wav) // CFG[\"hop\"]\n        rows.append(dict"
  "(\n            wav=(np.clip(wav, -1, 1) * 32767).astype(np.int16),\n            lab=build_labels(spans, nf, has_strong),\n            tags=clip_tags(ex),\n            quality=q, has_st"
  "rong=bool(has_strong),\n        ))\n        counts_done[q] += 1\n\n    if verbose:\n        print(counts_done, f\"| {len(rows)} rows | {(time.time()-t0)/60:.1f} min\")\n        if counts_d"
  "one.get(\"verified_timestamps\", 0) == 0:\n            print(\"!! ZERO gold - detection will be weak and the Track 2 banks thin\")\n    return rows\n\n\n# ----------------------------------"
  "-----------------------------------------\n# Tier bucketing + per-tier loss weights\n# ---------------------------------------------------------------------------\n# Per-sample loss w"
  "eight by tier (gold full, silver down-weighted, bronze tiny).\nQW = {\"verified_timestamps\": 1.0,\n      \"unverified_timestamps\": CFG[\"silver_weight\"],\n      \"no_timestamps\": 0.2}\n\n\nd"
  "ef bucket_by_tier(rows, verbose: bool = True):\n    \"\"\"Split the in-memory rows into (gold, silver, bronze) lists.\"\"\"\n    by_q = defaultdict(list)\n    for r in rows:\n        by_q[r["
  "\"quality\"]].append(r)\n    gold = by_q[\"verified_timestamps\"]      # trustworthy onset/offset\n    silver = by_q[\"unverified_timestamps\"]  # timestamps without agreement\n    bronze ="
  " by_q[\"no_timestamps\"]          # clip-level tags only\n    if verbose:\n        print(f\"gold {len(gold)} | silver {len(silver)} | bronze {len(bronze)}\")\n    return gold, silver, bro"
  "nze\n\n\n# ---------------------------------------------------------------------------\n# Windowing helpers used by the dataset\n# ------------------------------------------------------"
  "---------------------\ndef prep_wav(w):\n    \"\"\"Per-clip standardisation (zero mean, unit std). Applied identically at train\n    and inference time so the encoder sees the same distr"
  "ibution in both.\"\"\"\n    w = np.asarray(w, dtype=np.float32)\n    return (w - w.mean()) / (w.std() + 1e-5)\n\n\ndef _labels_20ms(lab, s):\n    \"\"\"Crop the 100 Hz label grid to the chosen"
  " window then max-pool 10 ms -> 20 ms\n    so the label rate matches the model's FRAMES_OUT output rate.\"\"\"\n    lab = lab[:, s // CFG[\"hop\"]: s // CFG[\"hop\"] + FRAMES_100HZ]\n    if l"
  "ab.shape[1] < FRAMES_100HZ:\n        lab = np.pad(lab, ((0, 0), (0, FRAMES_100HZ - lab.shape[1])))\n    lab = lab[:, :FRAMES_100HZ]\n    return lab.reshape(lab.shape[0], FRAMES_OUT, C"
  "FG[\"time_pool\"]).max(axis=2)\n\n\nclass SEDData(Dataset):\n    \"\"\"Fixed 5 s window dataset over the in-memory rows.\n\n    Each item returns wav/lab/tags/mask/strong/w tensors. A subtle "
  "correctness fix:\n    a random crop can cut the event out of the window while the stored clip tag still\n    says it is present. For timestamped clips we recompute the tag from the c"
  "ropped\n    labels so the weak loss is not poisoned by that contradiction.\n    \"\"\"\n\n    def __init__(self, rows, train=True):\n        self.rows, self.train = rows, train\n\n    def __"
  "len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, i):\n        m = self.rows[i]\n        wav = m[\"wav\"].astype(np.float32) / 32767.0\n        lab = m[\"lab\"].astype"
  "(np.float32)          # (N_OUT, T100)\n        tags = m[\"tags\"].astype(np.float32)        # (N_OUT,)\n\n        # Crop (train: random, val: from 0) or pad to a fixed 5 s window.\n     "
  "   if len(wav) > N_SAMP:\n            s = random.randint(0, len(wav) - N_SAMP) if self.train else 0\n            wav = wav[s:s + N_SAMP]\n            valid = N_SAMP\n        else:\n    "
  "        s = 0\n            valid = len(wav)\n            wav = np.pad(wav, (0, N_SAMP - len(wav)))\n\n        lab20 = _labels_20ms(lab, s)               # (N_OUT, FRAMES_OUT)\n        s"
  "trong = 1.0 if m[\"has_strong\"] else 0.0\n        # Recompute clip tag from the cropped labels for timestamped clips.\n        if m[\"has_strong\"]:\n            tags = (lab20.max(axis=1"
  ") > 0).astype(np.float32)\n\n        # Frame validity mask (1 for real audio frames, 0 for padding).\n        mask = np.zeros(FRAMES_OUT, dtype=np.float32)\n        n_valid = min(FRAME"
  "S_OUT, max(1, int(math.ceil(valid / CFG[\"sr\"] / FRAME_SEC))))\n        mask[:n_valid] = 1.0\n\n        return dict(\n            wav=torch.from_numpy(prep_wav(wav)),\n            lab=to"
  "rch.from_numpy(lab20.astype(np.float32)),\n            tags=torch.from_numpy(tags),\n            mask=torch.from_numpy(mask),\n            strong=torch.tensor(strong, dtype=torch.floa"
  "t32),\n            w=torch.tensor(float(QW.get(m[\"quality\"], 1.0)), dtype=torch.float32),\n        )\n\n\ndef split_and_load(gold, silver, bronze, t1_bs, train_limit=None, verbose=True)"
  ":\n    \"\"\"Build the train/val DataLoaders.\n\n    The validation set is gold-only (the only trustworthy strong labels). Everything\n    else (leftover gold + all silver + all bronze) t"
  "rains. `train_limit` optionally\n    caps the training set for smoke runs.\n    \"\"\"\n    random.Random(CFG[\"seed\"]).shuffle(gold)\n    n_val = max(1, int(len(gold) * CFG[\"val_frac\"]))\n"
  "    val_rows = gold[:n_val]\n    train_rows = gold[n_val:] + silver + bronze\n    random.Random(CFG[\"seed\"]).shuffle(train_rows)\n    if train_limit is not None:\n        train_rows = "
  "train_rows[:train_limit]\n    if verbose:\n        print(f\"train {len(train_rows)} | val {len(val_rows)} (gold only)\")\n\n    train_dl = DataLoader(SEDData(train_rows, True), batch_siz"
  "e=t1_bs, shuffle=True,\n                          num_workers=CFG[\"num_workers\"], drop_last=True, pin_memory=True)\n    val_dl = DataLoader(SEDData(val_rows, False), batch_size=t1_bs"
  ", shuffle=False,\n                        num_workers=CFG[\"num_workers\"], pin_memory=True)\n    return train_dl, val_dl\n"
),
  'model.py': (
  "\"\"\"Model definition for the v-3 WavLM SED baseline.\n\n`WavLMSED` is a WavLM-base+ encoder (last two transformer layers dropped) followed\nby a BiGRU and two heads:\n\n  * frame head  -"
  " per-frame event probabilities (the detection output).\n  * attention head - softmax over time producing an attention-pooled clip\n    probability. This is Multiple-Instance Learning"
  ": it is the only way Bronze\n    clips (clip-level tags, no timing) can contribute a gradient.\n\nWavLM emits 50 Hz frames, exactly the 20 ms label grid, so no interpolation is\nneeded"
  " during training - only a small safety interpolation for conv stride rounding.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functiona"
  "l as F\n\nfrom config import ENCODER_NAME, DROP_LAST_LAYERS, N_OUT\n\n\nclass WavLMSED(nn.Module):\n    \"\"\"WavLM encoder + BiGRU + frame/attention heads.\n\n    Args:\n        name: HF mode"
  "l id for the WavLM encoder.\n        n_out: number of output channels (8 = any-noise + 7 categories).\n        rnn_dim: hidden size per GRU direction.\n        drop_layers: number of "
  "final transformer layers to drop (regularisation).\n    \"\"\"\n\n    input_key = \"wav\"   # which batch key feeds the encoder (used by generic loops)\n\n    def __init__(self, name=ENCODER"
  "_NAME, n_out=N_OUT, rnn_dim=256,\n                 drop_layers=DROP_LAST_LAYERS):\n        super().__init__()\n        from transformers import AutoModel\n        self.enc = AutoModel."
  "from_pretrained(name)\n        if drop_layers > 0:\n            # Drop the final transformer blocks - they overfit the pretrain task.\n            self.enc.encoder.layers = self.enc.e"
  "ncoder.layers[:-drop_layers]\n        d = self.enc.config.hidden_size\n        self.rnn = nn.GRU(d, rnn_dim, 2, batch_first=True,\n                          bidirectional=True, dropou"
  "t=0.1)\n        self.strong = nn.Linear(2 * rnn_dim, n_out)   # frame head\n        self.att = nn.Linear(2 * rnn_dim, n_out)      # attention head\n\n    def forward(self, x, mask=None"
  "):\n        \"\"\"Return (frame, clip):\n          frame - (B, n_out, T) per-frame probabilities\n          clip  - (B, n_out)    attention-pooled clip probabilities\n        \"\"\"\n        "
  "h = self.enc(x).last_hidden_state             # (B, T_enc, d)\n        T = mask.shape[1] if mask is not None else h.shape[1]\n        if h.shape[1] != T:\n            # Conv stride ro"
  "unding can leave the sequence off by a frame or two;\n            # align it to the label length with linear interpolation.\n            h = F.interpolate(h.transpose(1, 2), size=T, "
  "mode=\"linear\",\n                              align_corners=False).transpose(1, 2)\n        h, _ = self.rnn(h)\n        frame = torch.sigmoid(self.strong(h))         # (B, T, n_out)\n "
  "       a = self.att(h)                               # (B, T, n_out) attention logits\n        if mask is not None:\n            # Never let padding frames win the softmax.\n         "
  "   a = a.masked_fill(mask.unsqueeze(-1) < 0.5, -1e4)\n        a = torch.softmax(a, dim=1)\n        # Attention-pooled clip probability: weight frame probs by attention over time.\n   "
  "     clip = (frame * a).sum(dim=1).clamp(1e-6, 1 - 1e-6)\n        return frame.transpose(1, 2), clip           # (B, n_out, T), (B, n_out)\n\n\ndef core_model(**kwargs):\n    \"\"\"Factory"
  " returning a fresh WavLMSED (keeps main.py free of import details).\"\"\"\n    return WavLMSED(**kwargs)\n"
),
  'losses.py': (
  "\"\"\"Loss terms and mean-teacher utilities for the v-3 WavLM baseline.\n\nThe per-step training loss combines three signals (see train.py for how they are\nassembled):\n\n  l_strong  - fr"
  "ame-level BCE, gold+silver only (the only timing supervision).\n  l_weak    - clip-tag BCE on the attention-pooled probability, all tiers (MIL;\n              Bronze's main path).\n  "
  "l_cons    - mean-teacher consistency between the augmented student and the clean\n              EMA teacher, tier-agnostic.\n\nThis module provides the primitive pieces: positive-weig"
  "hted BCE, the consistency\nramp-up schedule, augmentation/mixup for the student, and the EMA weight update.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport random\n\nimport numpy as n"
  "p\nimport torch\n\nfrom config import CFG\n\n\ndef bce_pos(p, target, pw=None):\n    \"\"\"Positive-weighted binary cross-entropy on probabilities (not logits).\n\n    Noise frames are a small"
  " minority, so plain BCE under-fires; `pw` (default\n    CFG['pos_weight']) up-weights the positive term. Inputs are clamped to avoid\n    log(0); this is computed in fp32 outside aut"
  "ocast because BCE on fp16 sigmoids\n    can saturate to exactly 1.0.\n    \"\"\"\n    pw = CFG[\"pos_weight\"] if pw is None else pw\n    p = p.clamp(1e-6, 1 - 1e-6)\n    return -(pw * targe"
  "t * torch.log(p) + (1 - target) * torch.log(1 - p))\n\n\ndef rampup(ep, n):\n    \"\"\"Sigmoid-shaped consistency ramp-up: 0 at epoch 0, ~1 by epoch `n`.\n\n    The consistency term must no"
  "t dominate before the teacher has stabilised, so its\n    weight ramps up over the first `n` epochs.\n    \"\"\"\n    return 1.0 if n == 0 else float(np.exp(-5 * (1 - np.clip(ep / n, 0, "
  "1)) ** 2))\n\n\ndef wav_augment(x, n=2, tmax=None):\n    \"\"\"Time-masking augmentation for the student: zero out up to `n` short spans.\n\n    Operates on a copy so the teacher still sees"
  " the clean waveform.\n    \"\"\"\n    tmax = tmax or int(0.15 * CFG[\"sr\"])\n    x = x.clone()\n    for b_ in range(x.shape[0]):\n        for _ in range(n):\n            t = random.randint(0"
  ", tmax)\n            t0 = random.randint(0, max(0, x.shape[1] - t))\n            x[b_, t0:t0 + t] = 0\n    return x\n\n\ndef mixup(inp, lab, tags, alpha=CFG[\"mixup_alpha\"]):\n    \"\"\"Mixup"
  " with UNION labels.\n\n    A mixed clip genuinely contains both event sets, so labels/tags combine with\n    `max` (not a soft blend). `lam` is pushed toward the larger side so the do"
  "minant\n    clip keeps its identity.\n    \"\"\"\n    lam = float(np.random.beta(alpha, alpha))\n    lam = max(lam, 1 - lam)\n    perm = torch.randperm(inp.size(0), device=inp.device)\n    "
  "return (lam * inp + (1 - lam) * inp[perm],\n            torch.maximum(lab, lab[perm]),\n            torch.maximum(tags, tags[perm]))\n\n\n@torch.no_grad()\ndef ema_update(stu, tea, decay"
  ", step=None):\n    \"\"\"Exponential-moving-average update of the teacher from the student.\n\n    The `step`-based cap `min(1 - 1/(step+1), decay)` ramps the effective decay up\n    from"
  " ~0, otherwise the teacher would sit near random init for the first ~1000\n    steps - exactly while the consistency weight is ramping up. Non-float buffers\n    (e.g. batch-norm cou"
  "nts) are copied verbatim.\n    \"\"\"\n    if step is not None:\n        decay = min(1 - 1 / (step + 1), decay)\n    for ts, ss in zip(tea.state_dict().values(), stu.state_dict().values()"
  "):\n        if ts.dtype.is_floating_point:\n            ts.mul_(decay).add_(ss.detach(), alpha=1 - decay)\n        else:\n            ts.copy_(ss)\n"
),
  'train.py': (
  "\"\"\"Two-stage mean-teacher training for the v-3 WavLM baseline.\n\nStage 1 freezes the encoder and trains only the GRU + heads (skipping this stage is\nthe classic failure mode: a larg"
  "e pretrained encoder fine-tuned end-to-end from step\n0 overfits immediately). Stage 2 unfreezes with a much smaller LR on the encoder.\n\nEvery step combines three losses (see losses"
  ".py and the notebook appendix):\n  loss = l_strong + l_weak + cw * l_cons\n  * l_strong - frame BCE, gold+silver only (masked by `strong`, weighted by `w`).\n  * l_weak   - clip-tag B"
  "CE on the attention-pooled prob, all tiers (Bronze's path).\n  * l_cons   - mean-teacher consistency between augmented student and clean teacher.\n\"\"\"\n\nfrom __future__ import annotat"
  "ions\n\nimport copy\nimport time\nfrom collections import defaultdict\n\nimport torch\nimport torch.nn as nn\nfrom torch.amp import autocast, GradScaler\n\nfrom config import CFG, CKPT, LR_E"
  "NC, LR_HEAD\nfrom model import WavLMSED\nfrom losses import bce_pos, rampup, wav_augment, mixup, ema_update\nfrom evaluate import evaluate\n\n\ndef run_epochs(stu, tea, opt, sched, epoch"
  "s, scaler, train_dl, val_dl, device,\n               state, tag=\"\"):\n    \"\"\"Run `epochs` of mean-teacher training, updating `state` (best score + gstep).\n\n    `state` is a shared di"
  "ct {\"best\": float, \"gstep\": int} so the two stages share a\n    global step counter (which drives the EMA decay ramp) and a single best checkpoint.\n    Saves the better of student/t"
  "eacher whenever the val Combined improves.\n    \"\"\"\n    for ep in range(epochs):\n        stu.train(); tea.train()\n        cw = CFG[\"max_cons_w\"] * rampup(ep, CFG[\"rampup_epochs\"])  "
  " # consistency weight\n        acc = defaultdict(float); t0 = time.time()\n\n        for batch in train_dl:\n            inp = batch[\"wav\"].to(device, non_blocking=True)\n            la"
  "b = batch[\"lab\"].to(device)\n            tags = batch[\"tags\"].to(device)\n            mask = batch[\"mask\"].to(device)\n            strong = batch[\"strong\"].to(device)\n            w = "
  "batch[\"w\"].to(device)\n\n            # Mixup with union labels (see losses.mixup).\n            if torch.rand(1).item() < CFG[\"mixup_prob\"]:\n                inp, lab, tags = mixup(inp"
  ", lab, tags)\n\n            # Student sees an augmented view; teacher sees the clean view (no grad).\n            with autocast(\"cuda\"):\n                f_s, c_s = stu(wav_augment(inp"
  "), mask)\n                with torch.no_grad():\n                    f_t, c_t = tea(inp, mask)\n\n            # Losses computed in fp32 outside autocast (BCE on fp16 sigmoids is unsafe"
  ").\n            m3 = mask.unsqueeze(1)\n            f_s = f_s.float().clamp(1e-6, 1 - 1e-6)\n            c_s = c_s.float().clamp(1e-6, 1 - 1e-6)\n\n            # Channel weights: ch0 (g"
  "raded any-noise) always 1.0; CFG[\"cat_w\"]\n            # scales auxiliary category channels 1..7 (T1.2 ablation).\n            # l_cons intentionally unweighted (alignment, not super"
  "vision).\n            chw = torch.ones(lab.shape[1], device=lab.device)\n            chw[1:] = CFG[\"cat_w\"]\n\n            # l_strong: per-frame BCE averaged over valid frames, then a "
  "weighted mean\n            # over only timestamped clips (`* strong` zeros Bronze, `* w` weights gold/silver).\n            ls = (bce_pos(f_s, lab) * chw.view(1, -1, 1) * m3).sum(dim"
  "=(1, 2)) / (m3.sum(dim=(1, 2)) * lab.shape[1] + 1e-6)\n            l_strong = (ls * strong * w).sum() / (strong * w).sum().clamp(min=1e-6)\n\n            # l_weak: clip-tag BCE on the"
  " attention-pooled prob (all tiers -> Bronze in).\n            l_weak = (bce_pos(c_s, tags) * chw.view(1, -1)).mean()\n\n            # l_cons: frame + clip MSE between student and teac"
  "her (tier-agnostic).\n            f_t, c_t = f_t.float(), c_t.float()\n            l_cons = (((f_s - f_t) ** 2) * m3).sum() / (m3.sum() * lab.shape[1] + 1e-6) \\\n                     "
  "+ ((c_s - c_t) ** 2).mean()\n\n            loss = l_strong + l_weak + cw * l_cons\n\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            s"
  "caler.unscale_(opt)\n            nn.utils.clip_grad_norm_(stu.parameters(), 5.0)\n            scaler.step(opt)\n            scaler.update()\n            if sched is not None:\n         "
  "       sched.step()\n\n            state[\"gstep\"] += 1\n            ema_update(stu, tea, CFG[\"ema_decay\"], step=state[\"gstep\"])\n\n            acc[\"s\"] += float(l_strong.detach()); acc["
  "\"w\"] += float(l_weak.detach())\n            acc[\"c\"] += float(l_cons.detach()); acc[\"n\"] += 1\n\n        # Validate both student and teacher; save the better if it improves.\n        n"
  " = max(1, acc[\"n\"])\n        ms = evaluate(stu, val_dl, device)\n        mt = evaluate(tea, val_dl, device)\n        print(f\"{tag}ep{ep:02d} [{(time.time()-t0)/60:.1f}m] s {acc['s']/n"
  ":.3f} \"\n              f\"w {acc['w']/n:.3f} c {acc['c']/n:.3f} | \"\n              f\"stu {ms['score']:.4f} (F1 {ms['f1']:.3f} D {ms['dice']:.3f}) | \"\n              f\"tea {mt['score']:"
  ".4f}\")\n        if max(ms[\"score\"], mt[\"score\"]) > state[\"best\"]:\n            state[\"best\"] = max(ms[\"score\"], mt[\"score\"])\n            which = \"student\" if ms[\"score\"] >= mt[\"score"
  "\"] else \"teacher\"\n            torch.save({\"model\": (stu if which == \"student\" else tea).state_dict(),\n                        \"which\": which, \"score\": state[\"best\"], \"cfg\": CFG}, C"
  "KPT)\n            print(f\"   saved {which} {state['best']:.4f}\")\n\n\ndef train(train_dl, val_dl, device, stage1_epochs, stage2_epochs):\n    \"\"\"Full two-stage training. Returns the sha"
  "red `state` dict (best score + gstep).\"\"\"\n    student = WavLMSED().to(device)\n    teacher = copy.deepcopy(student).to(device)\n    for p in teacher.parameters():\n        p.requires_"
  "grad_(False)\n    scaler = GradScaler(\"cuda\")\n    head = [p for n_, p in student.named_parameters() if not n_.startswith(\"enc.\")]\n    state = dict(best=-1.0, gstep=0)\n\n    # --- Sta"
  "ge 1: encoder frozen, train GRU + heads only ---\n    print(f\"=== stage 1: encoder frozen, {stage1_epochs} epochs ===\")\n    for p in student.enc.parameters():\n        p.requires_gra"
  "d_(False)\n    opt1 = torch.optim.AdamW(head, lr=LR_HEAD, weight_decay=CFG[\"weight_decay\"])\n    run_epochs(student, teacher, opt1, None, stage1_epochs, scaler,\n               train_"
  "dl, val_dl, device, state, tag=\"s1 \")\n\n    # --- Stage 2: unfreeze encoder with a small LR, cosine schedule ---\n    print(f\"=== stage 2: unfrozen, enc lr {LR_ENC}, {stage2_epochs} "
  "epochs ===\")\n    for p in student.enc.parameters():\n        p.requires_grad_(True)\n    opt2 = torch.optim.AdamW([{\"params\": student.enc.parameters(), \"lr\": LR_ENC},\n               "
  "               {\"params\": head, \"lr\": LR_HEAD * 0.3}],\n                             weight_decay=CFG[\"weight_decay\"])\n    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(\n     "
  "   opt2, T_max=stage2_epochs * max(1, len(train_dl)))\n    run_epochs(student, teacher, opt2, sched2, stage2_epochs, scaler,\n               train_dl, val_dl, device, state, tag=\"s2 "
  "\")\n\n    print(\"\\nbest Combined:\", state[\"best\"], \"/ 2.0\")\n    del student, teacher\n    torch.cuda.empty_cache()\n    return state\n"
),
  'evaluate.py': (
  "\"\"\"Official scorer metrics and cached evaluation for the v-3 WavLM baseline.\n\nThese functions are copied verbatim from the Track 1 Evaluation page so that local\nscores match the le"
  "aderboard exactly - do NOT \"improve\" them.\n\nKey scoring subtleties that drive tuning:\n  * event-match tolerance is `max(0.20 * duration, 0.05)` - a 50 ms floor.\n  * matching is glo"
  "bal closest-first (not first-fit).\n  * Segment Dice is macro per clip, and a clip with no reference AND no prediction\n    scores 1.0 - so one false alarm on a clean clip costs that"
  " clip's whole point.\n\nCombined = Event F1 + Segment Dice (max 2.0).\n\n`cache_posteriors` runs the model once; `score_cached` then scores any number of\npost-processing configs on the"
  " cached probabilities without re-running the network.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nfrom scipy.ndimage import median_filter\n\nfrom config "
  "import FRAME_SEC\n\n\n# ---------------------------------------------------------------------------\n# Event-based F1\n# ----------------------------------------------------------------"
  "-----------\ndef match_events(ref_events, pred_events, tolerance_frac=0.20):\n    \"\"\"Global closest-first matching of predicted to reference events.\n\n    A prediction matches a refer"
  "ence if both its onset and offset are within\n    `max(tolerance_frac * ref_duration, 0.05)` seconds. Among all valid candidate\n    pairs, the closest are matched first (greedy over"
  " the sorted distance list).\n    Returns (true_positives, false_positives, false_negatives).\n    \"\"\"\n    matched_ref, matched_pred, candidates = set(), set(), []\n    for ri, (r_on, "
  "r_off) in enumerate(ref_events):\n        tol = max(tolerance_frac * (r_off - r_on), 0.05)\n        for pi, (p_on, p_off) in enumerate(pred_events):\n            if abs(p_on - r_on) <"
  "= tol and abs(p_off - r_off) <= tol:\n                candidates.append((abs(p_on - r_on) + abs(p_off - r_off), ri, pi))\n    for _, ri, pi in sorted(candidates):\n        if ri not i"
  "n matched_ref and pi not in matched_pred:\n            matched_ref.add(ri)\n            matched_pred.add(pi)\n    tp = len(matched_ref)\n    return tp, len(pred_events) - tp, len(ref_e"
  "vents) - tp\n\n\ndef event_based_f1(ref_data, pred_data):\n    \"\"\"Micro-averaged event F1 across all clips. Predicted events for clip_ids not in\n    the reference count as pure false p"
  "ositives. Returns (f1, prec, rec, TP, FP, FN).\"\"\"\n    TP = FP = FN = 0\n    for cid, ref_events in ref_data.items():\n        tp, fp, fn = match_events(ref_events, pred_data.get(cid,"
  " []))\n        TP += tp; FP += fp; FN += fn\n    for cid in pred_data:\n        if cid not in ref_data:\n            FP += len(pred_data[cid])\n    p = TP / (TP + FP) if (TP + FP) else "
  "0.0\n    r = TP / (TP + FN) if (TP + FN) else 0.0\n    return ((2 * p * r / (p + r)) if (p + r) else 0.0), p, r, TP, FP, FN\n\n\n# ------------------------------------------------------"
  "---------------------\n# Segment Dice\n# ---------------------------------------------------------------------------\ndef events_to_frames(events, max_time, frame_len=0.01):\n    \"\"\"Ra"
  "sterise a list of (onset, offset) events to a binary 10 ms occupancy mask.\"\"\"\n    n = int(max_time / frame_len) + 1\n    mask = [0] * n\n    for on, off in events:\n        for i in r"
  "ange(int(on / frame_len), min(int(off / frame_len) + 1, n)):\n            mask[i] = 1\n    return mask\n\n\ndef segment_dice(ref_data, pred_data):\n    \"\"\"Macro-averaged per-clip Dice ov"
  "er 10 ms frames. A clip with no reference and\n    no prediction scores 1.0 (which is why the clip gate is valuable).\"\"\"\n    scores = []\n    for cid, ref_events in ref_data.items():"
  "\n        pred_events = pred_data.get(cid, [])\n        all_ev = ref_events + pred_events\n        if not all_ev:\n            scores.append(1.0)\n            continue\n        max_time "
  "= max(off for _, off in all_ev) + 0.5\n        rm = events_to_frames(ref_events, max_time)\n        pm = events_to_frames(pred_events, max_time)\n        inter = sum(r & p for r, p in"
  " zip(rm, pm))\n        total = sum(rm) + sum(pm)\n        scores.append(1.0 if total == 0 else 2.0 * inter / total)\n    return sum(scores) / len(scores) if scores else 0.0\n\n\n# ------"
  "---------------------------------------------------------------------\n# Posterior -> events + cached evaluation\n# ------------------------------------------------------------------"
  "---------\ndef prob_to_events(p, thr=0.5, med=7, frame_sec=FRAME_SEC, min_dur=0.05, merge_gap=0.10):\n    \"\"\"Turn a 1-D probability track into (onset, offset) events.\n\n    Steps: thr"
  "eshold -> optional median filter (de-flicker) -> extract runs ->\n    merge events closer than `merge_gap` -> drop events shorter than `min_dur`.\n    \"\"\"\n    b_ = (np.asarray(p) >= "
  "thr).astype(np.uint8)\n    if med > 1:\n        b_ = median_filter(b_, size=med, mode=\"nearest\")\n    ev, start = [], None\n    for i, v in enumerate(b_):\n        if v and start is Non"
  "e:\n            start = i\n        elif not v and start is not None:\n            ev.append([start * frame_sec, i * frame_sec])\n            start = None\n    if start is not None:\n    "
  "    ev.append([start * frame_sec, len(b_) * frame_sec])\n    out = []\n    for e in ev:\n        if out and e[0] - out[-1][1] <= merge_gap:\n            out[-1][1] = e[1]              "
  "        # merge across a short gap\n        else:\n            out.append(e)\n    return [e for e in out if e[1] - e[0] >= min_dur]\n\n\n@torch.no_grad()\ndef cache_posteriors(model, dl, "
  "device):\n    \"\"\"Run the model ONCE over a loader and cache (pred_prob, gt, clip_prob) per clip.\n\n    Posteriors do not change when sweeping post-processing, so caching lets us scor"
  "e\n    hundreds of configs cheaply. Also returns the ground-truth `ref` event dict.\n    \"\"\"\n    model.eval()\n    cache = []\n    for batch in dl:\n        frame, clip = model(batch[\"w"
  "av\"].to(device), batch[\"mask\"].to(device))\n        for p, cp, g, msk in zip(frame[:, 0].float().cpu().numpy(),\n                                 clip[:, 0].float().cpu().numpy(),\n  "
  "                               batch[\"lab\"][:, 0].numpy(),\n                                 batch[\"mask\"].numpy()):\n            nv = int(msk.sum())                    # keep only v"
  "alid (non-pad) frames\n            cache.append((p[:nv].copy(), g[:nv].copy(), float(cp)))\n    ref = {f\"c{i}\": [tuple(e) for e in prob_to_events(g, 0.5, 1, min_dur=0.0, merge_gap=0."
  "0)]\n           for i, (_, g, _) in enumerate(cache)}\n    return cache, ref\n\n\ndef score_cached(cache, ref, thr=0.5, med=7, min_dur=0.05, merge_gap=0.10, clip_gate=0.0):\n    \"\"\"Score"
  " one post-processing config against the cached posteriors.\n\n    `clip_gate`: if the attention-pooled clip probability is below this, emit NO\n    events for that clip. Because a cle"
  "an clip with no prediction scores a full Dice\n    point, staying silent when confident buys more than suppressing a real event\n    costs - so the gate is swept alongside thr/med/mi"
  "n_dur.\n    \"\"\"\n    pred = {}\n    for i, (p, _, cp) in enumerate(cache):\n        pred[f\"c{i}\"] = ([] if cp < clip_gate else\n                         [tuple(e) for e in prob_to_event"
  "s(p, thr, med, min_dur=min_dur,\n                                                           merge_gap=merge_gap)])\n    f1, prec, rec, TP, FP, FN = event_based_f1(ref, pred)\n    dice"
  " = segment_dice(ref, pred)\n    return dict(f1=f1, dice=dice, score=f1 + dice, precision=prec, recall=rec,\n                tp=TP, fp=FP, fn=FN)\n\n\ndef evaluate(model, dl, device, **k"
  "w):\n    \"\"\"Convenience: cache posteriors then score with one config.\"\"\"\n    cache, ref = cache_posteriors(model, dl, device)\n    return score_cached(cache, ref, **kw)\n\n\ndef sweep_p"
  "ostprocessing(cache, ref, verbose=True):\n    \"\"\"Grid-search thr / median / min_dur / clip_gate over cached posteriors.\n\n    Returns (best_params, sorted_results) where best_params "
  "is\n    (thr, med, min_dur, gate).\n    \"\"\"\n    results = []\n    for thr in [0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]:\n        for med in [1, 3, 5, 9, 15, 21]:\n            for min_dur in"
  " [0.05, 0.12, 0.25]:\n                for gate in [0.0, 0.3, 0.5, 0.7]:\n                    m = score_cached(cache, ref, thr=thr, med=med,\n                                     min_d"
  "ur=min_dur, clip_gate=gate)\n                    results.append((m[\"score\"], m[\"f1\"], m[\"dice\"], thr, med, min_dur,\n                                    gate, m[\"precision\"], m[\"reca"
  "ll\"]))\n    results.sort(reverse=True)\n    if verbose:\n        print(f\"{'Comb':>7}{'F1':>7}{'Dice':>7}{'thr':>6}{'med':>5}\"\n              f\"{'mind':>7}{'gate':>6}{'P':>7}{'R':>7}\")\n"
  "        for r in results[:12]:\n            print(f\"{r[0]:7.4f}{r[1]:7.3f}{r[2]:7.3f}{r[3]:6.2f}{r[4]:5d}{r[5]:7.2f}\"\n                  f\"{r[6]:6.2f}{r[7]:7.3f}{r[8]:7.3f}\")\n    bes"
  "t = (results[0][3], results[0][4], results[0][5], results[0][6])\n    return best, results\n"
),
  'inference.py': (
  "\"\"\"Whole-clip inference and Track 1 submission export for the v-3 WavLM baseline.\n\n`t1_posteriors` runs a whole clip through the model. The conv front-end and GRU take\nany length, "
  "so no windowing is needed below MAX_ONESHOT_SEC - which also avoids a\ntrain/inference mismatch since normalisation is computed over the whole clip in both\nplaces. Longer clips are "
  "processed in overlapping 30 s windows and averaged.\n\n`export_track1` turns posteriors into onset/offset events using the swept\npost-processing params (threshold, median, min durati"
  "on, clip gate).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\nimport torch\n\nfrom config import CFG, FRAME_SEC, MIN_SAMPLES, MAX_ONESHOT_SEC, N_OUT\nfrom d"
  "ata import prep_wav\nfrom evaluate import prob_to_events\n\n\n@torch.no_grad()\ndef t1_posteriors(wav, model, device, sr=CFG[\"sr\"]):\n    \"\"\"Return (post (N_OUT, T20), clip_prob_any_nois"
  "e) for a whole clip.\n\n    Short clips are padded to MIN_SAMPLES. Clips up to MAX_ONESHOT_SEC go through in a\n    single pass; longer ones use overlapping 30 s windows (2 s hop) who"
  "se posteriors\n    are averaged frame-wise, with the clip prob taken as the max across windows.\n    \"\"\"\n    wav = np.asarray(wav, dtype=np.float32)\n    if len(wav) < MIN_SAMPLES:\n  "
  "      wav = np.pad(wav, (0, MIN_SAMPLES - len(wav)))\n    dur = len(wav) / sr\n    n_out = max(1, int(math.ceil(dur / FRAME_SEC)))\n\n    # --- single-pass path ---\n    if dur <= MAX_O"
  "NESHOT_SEC:\n        x = torch.from_numpy(prep_wav(wav)).unsqueeze(0).to(device)\n        fr, cl = model(x, None)\n        p = fr[0].float().cpu().numpy()\n        cp = float(cl[0, 0])"
  "\n        if p.shape[1] != n_out:                       # align to the clip's frame count\n            p = np.stack([np.interp(np.linspace(0, 1, n_out),\n                             "
  "       np.linspace(0, 1, p.shape[1]), r) for r in p])\n        return p, cp\n\n    # --- windowed path for long clips ---\n    win, hp = int(30 * sr), int(28 * sr)\n    starts = list(ra"
  "nge(0, max(1, len(wav) - win + hp), hp)) or [0]\n    acc = np.zeros((N_OUT, n_out)); cnt = np.zeros(n_out) + 1e-9; cps = []\n    for s0 in starts:\n        seg = wav[s0:s0 + win]\n    "
  "    if len(seg) < MIN_SAMPLES:\n            continue\n        x = torch.from_numpy(prep_wav(seg)).unsqueeze(0).to(device)\n        fr, cl = model(x, None)\n        p = fr[0].float().cp"
  "u().numpy(); cps.append(float(cl[0, 0]))\n        off = int(round(s0 / sr / FRAME_SEC))         # frame offset of this window\n        for j in range(p.shape[1]):\n            k = off"
  " + j\n            if 0 <= k < n_out:\n                acc[:, k] += p[:, j]; cnt[k] += 1\n    return acc / cnt, (max(cps) if cps else 0.0)\n\n\ndef export_track1(clip_id, wav, model, devi"
  "ce, thr, med, min_dur, gate, sr=CFG[\"sr\"]):\n    \"\"\"Produce the submission record for one clip: {clip_id, events:[{onset,offset}]}.\n\n    The clip gate short-circuits to no events wh"
  "en the model believes the clip is\n    clean (see evaluate.score_cached for why that pays off).\n    \"\"\"\n    post, clip_p = t1_posteriors(wav, model, device, sr)\n    dur = max(len(wa"
  "v) / sr, MIN_SAMPLES / sr)\n    spans = []\n    events_iter = [] if clip_p < gate else prob_to_events(post[0], thr, med, min_dur=min_dur)\n    for on, off in events_iter:\n        on, "
  "off = float(max(0.0, on)), float(min(dur, off))\n        if off - on < 0.05:                           # drop sub-50 ms slivers\n            continue\n        spans.append([on, off])\n"
  "    return {\"clip_id\": clip_id,\n            \"events\": [{\"onset\": round(float(a), 3), \"offset\": round(float(b), 3)}\n                       for a, b in spans]}\n"
),
  'main.py': (
  "\"\"\"End-to-end orchestration for the v-3 WavLM SED baseline.\n\nRun order mirrors the original notebook:\n  1. Config + budget resolution + seeding.\n  2. Stream the dataset and decode/"
  "rasterise once into `rows`.\n  3. Bucket by tier, build train/val loaders.\n  4. Two-stage mean-teacher training (checkpoint saved to CKPT).\n  5. Post-processing sweep on cached val "
  "posteriors -> best (thr, med, min_dur, gate).\n  6. Optional: write the Track 1 submission ZIP from a test-audio folder.\n\nImport errors for torch/datasets/etc. on a local machine ar"
  "e expected - this is\ndesigned to run in the Kaggle GPU environment where those packages exist.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport zipfile\nfrom pathlib impo"
  "rt Path\n\nimport torch\n\nfrom config import (\n    BUDGET, CFG, CKPT, WORK, budget_settings, seed_everything,\n)\nimport data\nfrom model import WavLMSED\nfrom train import train\nfrom eva"
  "luate import cache_posteriors, sweep_postprocessing\nfrom inference import export_track1\n\n\nAUD = (\".wav\", \".flac\", \".mp3\", \".ogg\")\n\n\ndef read_audio(p, sr=CFG[\"sr\"]):\n    \"\"\"Load an "
  "audio file to mono float32 at the model sample rate.\"\"\"\n    import soundfile as sf\n    import librosa\n    w, s = sf.read(str(p), dtype=\"float32\", always_2d=False)\n    if w.ndim > 1"
  ":\n        w = w.mean(axis=1)\n    if s != sr:\n        w = librosa.resample(w, orig_sr=s, target_sr=sr)\n    return w\n\n\ndef write_submission(model, device, thr, med, min_dur, gate,\n  "
  "                   test_dir=Path(\"/kaggle/input/indoml-track1-test\")):\n    \"\"\"Write predictions.jsonl (+ zip) for every clip under `test_dir`.\n\n    Official format: ZIP with predic"
  "tions.jsonl at the ROOT; every eval clip appears\n    exactly once; clips with nothing detected get an empty events list, not a missing\n    line.\n    \"\"\"\n    from tqdm.auto import t"
  "qdm\n    jsonl, zip_path = WORK / \"predictions.jsonl\", WORK / \"submission_track1.zip\"\n    if not test_dir.exists():\n        print(f\"{test_dir} not found - attach the Track 1 test se"
  "t and re-run.\")\n        return\n    files = sorted([p for p in test_dir.rglob(\"*\") if p.suffix.lower() in AUD])\n    print(len(files), \"test clips\")\n    with open(jsonl, \"w\", encodin"
  "g=\"utf-8\") as f:\n        for p in tqdm(files):\n            rec = export_track1(p.stem, read_audio(p), model, device,\n                                thr, med, min_dur, gate)\n      "
  "      f.write(json.dumps({\"clip_id\": p.stem, \"events\": rec[\"events\"]},\n                               ensure_ascii=False) + \"\\n\")\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZI"
  "P_DEFLATED) as z:\n        z.write(jsonl, \"predictions.jsonl\")           # at ZIP root, no enclosing folder\n    n = sum(1 for _ in open(jsonl))\n    empty = sum(1 for l in open(jsonl"
  ") if not json.loads(l)[\"events\"])\n    print(f\"{n} lines ({empty} with no events) -> {zip_path}\")\n\n\ndef run(budget=BUDGET, hf_token=None, do_submission=False, train_limit=None):\n   "
  " \"\"\"Full pipeline. Returns (best_state, best_postproc_params).\n\n    `train_limit` overrides the budget default (fast: 100, full: None);\n    per-tier caps are tightened to just abov"
  "e it (see budget_settings).\n    \"\"\"\n    seed_everything()\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    data.seed_torch()\n\n    max_per_quality, stage1_epochs, sta"
  "ge2_epochs, t1_bs, train_limit = \\\n        budget_settings(budget, train_limit)\n\n    # --- data ---\n    raw = data.load_stream(hf_token)\n    rows = data.build_rows(raw, max_per_qua"
  "lity)\n    gold, silver, bronze = data.bucket_by_tier(rows)\n    train_dl, val_dl = data.split_and_load(gold, silver, bronze, t1_bs, train_limit)\n\n    # --- train ---\n    state = tra"
  "in(train_dl, val_dl, device, stage1_epochs, stage2_epochs)\n\n    # --- post-processing sweep on the best checkpoint ---\n    ck = torch.load(CKPT, map_location=device, weights_only=F"
  "alse)\n    model = WavLMSED().to(device)\n    model.load_state_dict(ck[\"model\"])\n    model.eval()\n    cache, ref = cache_posteriors(model, val_dl, device)\n    (thr, med, min_dur, gat"
  "e), _ = sweep_postprocessing(cache, ref)\n    ck.update(thr=float(thr), med=int(med), min_dur=float(min_dur), gate=float(gate))\n    torch.save(ck, CKPT)\n    print(f\"best post-proc: "
  "thr={thr} med={med} min_dur={min_dur} gate={gate}\")\n\n    # --- optional submission ---\n    if do_submission:\n        write_submission(model, device, thr, med, min_dur, gate)\n\n    r"
  "eturn state, (thr, med, min_dur, gate)\n\n\nif __name__ == \"__main__\":\n    run()\n"
),
}
_BASE = os.getcwd()
for _name, _src in _MODULES.items():
    _p = os.path.join(_BASE, _name)
    if not os.path.exists(_p):
        open(_p, "w", encoding="utf-8").write(_src)
print("modules:", sorted(_MODULES))
if _BASE not in sys.path:
    sys.path.insert(0, _BASE)
!pip -q install soundfile librosa huggingface_hub 2>&1 | tail -1
import torch
print("cuda:", torch.cuda.is_available())


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Attach Secret HF_TOKEN and re-run"
login(token=HF_TOKEN)
print("HF login ok")


In [ ]:
# --- stream once (identical data for all 3 configs) ---
import torch
from config import CFG, CKPT, budget_settings
import data
device = "cuda" if torch.cuda.is_available() else "cpu"
caps, s1, s2, bs, limit = budget_settings("fast", 2000)
raw = data.load_stream(HF_TOKEN)
rows = data.build_rows(raw, caps)
gold, silver, bronze = data.bucket_by_tier(rows)
train_dl, val_dl = data.split_and_load(gold, silver, bronze, bs, limit)
print("train/val:", len(train_dl.dataset), len(val_dl.dataset), flush=True)


In [ ]:
# --- T1.2: 3 configs, same loaders, separate checkpoints ---
import shutil
import torch
from model import WavLMSED
from train import train
from evaluate import cache_posteriors, sweep_postprocessing
RESULTS = []
for tag, catw in [("catw10", 1.0), ("catw05", 0.5), ("catw00", 0.0)]:
    CFG["cat_w"] = catw
    print(f"===== {tag}: cat_w={catw} =====", flush=True)
    state = train(train_dl, val_dl, device, s1, s2)
    dst = f"/kaggle/working/t1_t12_{tag}.pt"
    shutil.copy(str(CKPT), dst)
    ck = torch.load(dst, map_location=device, weights_only=False)
    model = WavLMSED().to(device)
    model.load_state_dict(ck["model"])
    model.eval()
    cache, ref = cache_posteriors(model, val_dl, device)
    (thr, med, mind, gate), allres = sweep_postprocessing(cache, ref)
    b = allres[0]
    print(f"{tag} BEST: comb={b[0]:.4f} F1={b[1]:.4f} Dice={b[2]:.4f} "
          f"thr={b[3]} med={b[4]} mind={b[5]} gate={b[6]} P={b[7]:.3f} R={b[8]:.3f}", flush=True)
    RESULTS.append((tag, catw, b))
    del model
    torch.cuda.empty_cache()
print("===== SUMMARY =====")
for tag, catw, b in RESULTS:
    print(f"{tag} cat_w={catw}: comb={b[0]:.4f} F1={b[1]:.4f} Dice={b[2]:.4f} "
          f"thr={b[3]} med={b[4]} mind={b[5]} gate={b[6]}")
